This notebook copies the content from https://bluetickconsultants.medium.com/react-integrating-reasoning-and-acting-with-retrieval-augmented-generation-rag-a6c2e869f763 <br>

* ReAct agent: llama_index.core.agent.legacy.react.base.ReActAgent <br>
* LLM: OpenAI <br>
* RAG: llama_index.core.VectorStoreIndex

In [1]:
!pip -q install llama-index-llms-openai llama-index-readers-file llama-index-embeddings-openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.9/378.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.1 MB/s eta 0:00:00


In [9]:
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    load_index_from_storage,
)

from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.core.agent.legacy.react.base import ReActAgent
from llama_index.llms.openai import OpenAI
import os
import openai

In [3]:
try:
    storage_context = StorageContext.from_defaults(
        persist_dir="./storage/RAG"
    )
    RAG_index = load_index_from_storage(storage_context)

    index_loaded = True
except:
    index_loaded = False

In [5]:
api_key=""
Settings.embed_model = OpenAIEmbedding(api_key=api_key)

In [6]:
if not index_loaded:
    RAG_docs = SimpleDirectoryReader(
        input_files=["/content/naf23.pdf"]
    ).load_data()

    RAG_index= VectorStoreIndex.from_documents(RAG_docs)
    RAG_index.storage_context.persist(persist_dir="./storage/RAG")

In [8]:

os.environ["OPENAI_API_KEY"] = api_key
openai.api_key = os.environ["OPENAI_API_KEY"]
# llm = OpenAI(temperature=0.9, model="gpt-4-turbo")
# llm = OpenAI(temperature=0.9, model=("gpt-4o"))
llm = OpenAI(temperature=0.1, model=("gpt-4o"))


In [11]:
RAG_index_engine=RAG_index.as_query_engine(similarity_top_k=3)
query_engine_tools = [
    QueryEngineTool(
        query_engine=RAG_index_engine,
        metadata=ToolMetadata(
            name="search",
            description=(
                "test"
            ),
        ),
    ),
]

In [12]:
context = """\
You are an expert on analysing financial statement and auditor report.
 """
agent = ReActAgent.from_tools(
    query_engine_tools,
    llm=llm,
    verbose=True,
    context=context

)

In [15]:
response = agent.chat("Can you analyse  National Ataxia Foudnations's current assets, provide potential drivers")
print(str(response))

Thought: The current language of the user is English. I need to use a tool to find the latest financial statements or reports of the National Ataxia Foundation.
Action: search
Action Input: {'input': 'National Ataxia Foundation latest financial statements 2023'}
Observation: The National Ataxia Foundation's latest financial statements for the year 2023 include the following:
- Independent Auditor's Report
- Statements of Financial Position
- Statements of Activities
- Statements of Functional Expenses
- Statements of Cash Flows
- Notes to the Financial Statements
Thought: I have the necessary documents to analyze the National Ataxia Foundation's current assets and identify potential drivers. I will focus on the Statements of Financial Position and the Notes to the Financial Statements to gather relevant information.
Action: search
Action Input: {'input': 'National Ataxia Foundation Statements of Financial Position 2023'}
Observation: The National Ataxia Foundation's Statements of Finan

In [13]:
response = agent.chat("National Ataxia Foudnations's total expense increased from 4,131,189 in 2022 to 4,184,787 in 2023, do you know why, what caused it")
print(str(response))

Thought: The current language of the user is English. I need to use a tool to help me answer the question.
Action: search
Action Input: {'input': 'National Ataxia Foundation total expense increase from 2022 to 2023 reasons'}
Observation: The National Ataxia Foundation's total expenses increased from 2022 to 2023 primarily due to higher salaries and related expenses, increased meeting expenses, and a rise in research grants. Additionally, there were elevated costs in professional services and supporting services like fundraising and management.
Thought: I can answer without using any more tools. I'll use the user's language to answer.
Answer: The National Ataxia Foundation's total expenses increased from 2022 to 2023 primarily due to higher salaries and related expenses, increased meeting expenses, and a rise in research grants. Additionally, there were elevated costs in professional services and supporting services like fundraising and management.
The National Ataxia Foundation's total

In [17]:
response = agent.chat("compare National Ataxia Foudnations's total revenue difference between 2022 and 2023, what changed")
print(str(response))

Thought: The current language of the user is English. I need to use a tool to help me find the specific revenue figures for the National Ataxia Foundation for the years 2022 and 2023.
Action: search
Action Input: {'input': 'National Ataxia Foundation total revenue 2022 and 2023'}
Observation: National Ataxia Foundation's total revenue for 2022 was $981,690 and for 2023 was $979,651.
Thought: I have the total revenue figures for the National Ataxia Foundation for both 2022 and 2023. Now, I can compare the differences and identify what changed.

The total revenue for 2022 was $981,690, and for 2023 it was $979,651. This indicates a slight decrease in total revenue by $2,039.

To understand what changed, we need to consider potential factors such as:

1. **Donor Contributions**: A decrease in donations from individuals, corporations, or foundations could have contributed to the lower revenue.
2. **Fundraising Events**: If fundraising events were less successful or fewer in number, this co

Observation:
* I do think llamaidnex has better RAG tool implemented
* I need more detailed audtior note tailored to the specific questions we want to investigate to test the performance of ReAct
* other observation is similar to ReAct using langchain